In [40]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pickle
import shap
import matplotlib.pyplot as plt
from matplotlib import font_manager

print("="*80)
print("🎯 审稿人验证程序：SHAP分析（103个特征）- 增强版")
print("="*80)
print("目的：计算103个特征的SHAP值，并查看RFE筛选的6个特征排名")
print("新增：Top 50特征蜂群图 + 完整可视化数据导出")
print("="*80)

# ====================== 1. 定义模型结构（必须与训练时完全一致）======================
class MyModel_103(nn.Module):
    """验证模型：103个特征"""
    def __init__(self, input_dim=103):
        super(MyModel_103, self).__init__()
        self.layer1 = nn.Linear(input_dim, 128)
        self.ln1 = nn.LayerNorm(128)
        self.relu1 = nn.ReLU()
        
        self.layer2 = nn.Linear(128, 64)
        self.ln2 = nn.LayerNorm(64)
        self.relu2 = nn.ReLU()
        
        self.layer3 = nn.Linear(64, 32)
        self.ln3 = nn.LayerNorm(32)
        self.relu3 = nn.ReLU()
        
        self.output = nn.Linear(32, 1)

    def forward(self, x):
        x = self.layer1(x)
        x = self.ln1(x)
        x = self.relu1(x)
        
        x = self.layer2(x)
        x = self.ln2(x)
        x = self.relu2(x)
        
        x = self.layer3(x)
        x = self.ln3(x)
        x = self.relu3(x)
        
        x = self.output(x)
        return x

# ====================== 2. 定义路径 ======================
# 模型路径
model_dir = "D:\\博一下\\manuscript3-机器学习\\nc\\nc修改9.27\\10.15\\10.21\\1.4程序补充\\验证分析_103特征"

# SHAP输出路径
shap_output_dir = "D:\\博一下\\manuscript3-机器学习\\nc\\nc修改9.27\\10.15\\10.21\\1.4程序补充\\SHAP分析_103特征1.17"
if not os.path.exists(shap_output_dir):
    os.makedirs(shap_output_dir)

print(f"\n路径设置:")
print(f"  模型路径: {model_dir}")
print(f"  SHAP输出路径: {shap_output_dir}")

# ====================== 3. RFE筛选出的6个特征（来自原始训练）======================
# 这是你原始训练程序中RFE筛选出的6个特征
RFE_SELECTED_FEATURES = [
    "ΔHmix",
    "mean_C1 temperature melting", 
    "mean_S13 radii atomic (coordination number 12) (pm)",
    "mean_热导率W/(mk)",
    "var_C1 temperature melting",
    "var_热导率W/(mk)"
]

print(f"\n🎯 RFE筛选的6个关键特征:")
for i, feat in enumerate(RFE_SELECTED_FEATURES, 1):
    print(f"  {i}. {feat}")

# ====================== 4. 加载数据 ======================
print("\n加载数据...")
df = pd.read_excel("D:\\文成数据库\\Nb-Si数据库3.4-成分-性能.xlsx")

# 加载特征列表
features_path = os.path.join(model_dir, 'features_103.txt')
with open(features_path, 'r', encoding='utf-8') as f:
    features_103 = [line.strip() for line in f.readlines()]

print(f"  特征数量: {len(features_103)}")

# 加载标准化器
scaler_path = os.path.join(model_dir, 'scaler_103.pkl')
with open(scaler_path, 'rb') as f:
    scaler_103 = pickle.load(f)
print(f"  ✅ 标准化器已加载")

# ====================== 5. 加载模型 ======================
print("\n加载模型...")
device = torch.device("cpu")  # 使用CPU更稳定

# 创建模型实例
model = MyModel_103(input_dim=103)

# 加载模型权重
model_weights_path = os.path.join(model_dir, 'model_103_weights.pth')
model.load_state_dict(torch.load(model_weights_path, map_location='cpu'))
print(f"  ✅ 模型权重已加载")

model.to(device)
model.eval()

# ====================== 6. 准备数据 ======================
print("\n准备SHAP分析数据...")
X_original = df[features_103].values  # 原始特征值
X_scaled = scaler_103.transform(X_original)  # 标准化特征值
y = df['KQ'].values

print(f"  样本数: {X_scaled.shape[0]}")
print(f"  特征数: {X_scaled.shape[1]}")

# ====================== 7. 创建SHAP模型包装器 ======================
class ShapModelWrapper:
    """SHAP模型包装器"""
    def __init__(self, model):
        self.model = model
        
    def __call__(self, X):
        if isinstance(X, np.ndarray):
            X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
        else:
            X_tensor = X
            
        with torch.no_grad():
            output = self.model(X_tensor)
            
        return output.cpu().numpy()

# ====================== 8. SHAP分析 - 使用Kernel解释器 ======================
print("\n开始SHAP分析...")
print("  这可能需要几分钟时间...")

# 创建背景数据集
max_bg_samples = min(100, X_scaled.shape[0])
bg_indices = np.random.choice(X_scaled.shape[0], max_bg_samples, replace=False)
bg_data = X_scaled[bg_indices]
print(f"  背景样本数: {max_bg_samples}")

# 创建SHAP解释器
model_wrapper = ShapModelWrapper(model)
explainer = shap.KernelExplainer(model_wrapper, bg_data)
print(f"  ✅ SHAP解释器已创建")

# 计算SHAP值（使用所有样本）
print(f"  正在计算所有 {X_scaled.shape[0]} 个样本的SHAP值...")
shap_values = explainer.shap_values(X_scaled)

# 处理SHAP值格式
if isinstance(shap_values, list) and len(shap_values) == 1:
    shap_values = shap_values[0]

if len(np.array(shap_values).shape) > 2:
    shap_values = np.squeeze(shap_values)

print(f"  ✅ SHAP值计算完成")
print(f"  SHAP值形状: {np.array(shap_values).shape}")

# ====================== 9. 计算特征重要性（平均绝对SHAP值）======================
print("\n计算特征重要性...")
mean_abs_shap = np.mean(np.abs(shap_values), axis=0)

# 创建特征重要性DataFrame
shap_importance_df = pd.DataFrame({
    'Feature': features_103,
    'Mean_Absolute_SHAP_Value': mean_abs_shap,
    'Relative_Importance_%': mean_abs_shap / np.sum(mean_abs_shap) * 100
})

# 按平均绝对SHAP值排序
shap_importance_df = shap_importance_df.sort_values('Mean_Absolute_SHAP_Value', ascending=False)
shap_importance_df['Rank'] = range(1, len(shap_importance_df) + 1)

# 保存特征重要性
importance_path = os.path.join(shap_output_dir, 'feature_importance_all_103.xlsx')
shap_importance_df.to_excel(importance_path, index=False)
print(f"  ✅ 特征重要性已保存: {importance_path}")

# ====================== 10. 🎯 关键分析：RFE的6个特征排在第几？======================
print("\n" + "="*80)
print("🎯 关键结果：RFE筛选的6个特征在103个特征中的SHAP排名")
print("="*80)

rfe_ranking_results = []

for rfe_feature in RFE_SELECTED_FEATURES:
    # 查找该特征的排名
    feature_row = shap_importance_df[shap_importance_df['Feature'] == rfe_feature]
    
    if not feature_row.empty:
        rank = int(feature_row['Rank'].values[0])
        shap_value = float(feature_row['Mean_Absolute_SHAP_Value'].values[0])
        relative_imp = float(feature_row['Relative_Importance_%'].values[0])
        
        rfe_ranking_results.append({
            'RFE_Feature': rfe_feature,
            'SHAP_Rank': rank,
            'Mean_Abs_SHAP': shap_value,
            'Relative_Importance_%': relative_imp
        })
        
        print(f"✓ {rfe_feature}")
        print(f"  排名: {rank} / 103")
        print(f"  平均绝对SHAP值: {shap_value:.6f}")
        print(f"  相对重要性: {relative_imp:.2f}%")
        print()
    else:
        print(f"⚠️ 未找到特征: {rfe_feature}")

# 保存RFE特征排名结果
rfe_ranking_df = pd.DataFrame(rfe_ranking_results)
rfe_ranking_df = rfe_ranking_df.sort_values('SHAP_Rank')
rfe_ranking_path = os.path.join(shap_output_dir, 'RFE_features_SHAP_ranking.xlsx')
rfe_ranking_df.to_excel(rfe_ranking_path, index=False)
print(f"✅ RFE特征排名已保存: {rfe_ranking_path}")

# 统计分析
print("\n" + "="*80)
print("📊 统计总结")
print("="*80)
ranks = [r['SHAP_Rank'] for r in rfe_ranking_results]
print(f"RFE的6个特征排名统计:")
print(f"  最高排名: {min(ranks)}")
print(f"  最低排名: {max(ranks)}")
print(f"  平均排名: {np.mean(ranks):.1f}")
print(f"  中位排名: {np.median(ranks):.1f}")

top_10_count = sum(1 for r in ranks if r <= 10)
top_20_count = sum(1 for r in ranks if r <= 20)
top_30_count = sum(1 for r in ranks if r <= 30)

print(f"\n排名分布:")
print(f"  Top 10: {top_10_count} / 6 特征 ({top_10_count/6*100:.1f}%)")
print(f"  Top 20: {top_20_count} / 6 特征 ({top_20_count/6*100:.1f}%)")
print(f"  Top 30: {top_30_count} / 6 特征 ({top_30_count/6*100:.1f}%)")

# ====================== 11. 🆕 Top 50特征蜂群图及数据导出 ======================
print("\n" + "="*80)
print("🆕 生成Top 50特征蜂群图及可视化数据")
print("="*80)

# 获取Top 50特征
top50_features = shap_importance_df.head(50)['Feature'].tolist()
top50_indices = [features_103.index(f) for f in top50_features]

# 提取Top 50特征的SHAP值和特征值
shap_values_top50 = shap_values[:, top50_indices]
X_original_top50 = X_original[:, top50_indices]

print(f"  Top 50特征已提取")
print(f"  SHAP值矩阵形状: {shap_values_top50.shape}")

# 保存蜂群图的原始数据到Excel
print("\n导出蜂群图可视化数据...")

# 为每个Top 50特征创建数据表
beeswarm_data_list = []

for i, feature_name in enumerate(top50_features):
    feature_data = pd.DataFrame({
        'Feature_Name': feature_name,
        'Feature_Rank': i + 1,  # 排名（1-50）
        'Sample_Index': range(len(X_original_top50)),
        'Feature_Value': X_original_top50[:, i],
        'SHAP_Value': shap_values_top50[:, i],
        'Abs_SHAP_Value': np.abs(shap_values_top50[:, i])
    })
    beeswarm_data_list.append(feature_data)

# 合并所有数据
beeswarm_all_data = pd.concat(beeswarm_data_list, ignore_index=True)

# 保存到Excel
beeswarm_data_path = os.path.join(shap_output_dir, 'beeswarm_plot_data_top50.xlsx')
beeswarm_all_data.to_excel(beeswarm_data_path, index=False)
print(f"  ✅ 蜂群图数据已保存: {beeswarm_data_path}")

# 额外保存每个特征的统计摘要
beeswarm_summary = []
for i, feature_name in enumerate(top50_features):
    summary = {
        'Rank': i + 1,
        'Feature': feature_name,
        'Mean_SHAP': np.mean(shap_values_top50[:, i]),
        'Std_SHAP': np.std(shap_values_top50[:, i]),
        'Min_SHAP': np.min(shap_values_top50[:, i]),
        'Max_SHAP': np.max(shap_values_top50[:, i]),
        'Mean_Abs_SHAP': np.mean(np.abs(shap_values_top50[:, i])),
        'Mean_Feature_Value': np.mean(X_original_top50[:, i]),
        'Std_Feature_Value': np.std(X_original_top50[:, i]),
        'Min_Feature_Value': np.min(X_original_top50[:, i]),
        'Max_Feature_Value': np.max(X_original_top50[:, i])
    }
    beeswarm_summary.append(summary)

beeswarm_summary_df = pd.DataFrame(beeswarm_summary)
beeswarm_summary_path = os.path.join(shap_output_dir, 'beeswarm_summary_statistics_top50.xlsx')
beeswarm_summary_df.to_excel(beeswarm_summary_path, index=False)
print(f"  ✅ 蜂群图统计摘要已保存: {beeswarm_summary_path}")

# ====================== 12. 可视化：蜂群图 ======================
print("\n创建蜂群图...")

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 创建蜂群图
fig, ax = plt.subplots(figsize=(12, 16))

# 使用SHAP的beeswarm plot
shap.summary_plot(
    shap_values_top50, 
    X_original_top50,
    feature_names=top50_features,
    show=False,
    max_display=50,
    plot_size=(12, 16)
)

plt.title('SHAP Beeswarm Plot - Top 50 Features', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('SHAP value (impact on model output)', fontsize=12)
plt.tight_layout()

beeswarm_path = os.path.join(shap_output_dir, 'shap_beeswarm_plot_top50.png')
plt.savefig(beeswarm_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✅ 蜂群图已保存: {beeswarm_path}")

# 创建高分辨率版本（用于论文）
fig, ax = plt.subplots(figsize=(14, 18))
shap.summary_plot(
    shap_values_top50, 
    X_original_top50,
    feature_names=top50_features,
    show=False,
    max_display=50,
    plot_size=(14, 18)
)
plt.title('SHAP Beeswarm Plot - Top 50 Features (High Resolution)', 
          fontsize=18, fontweight='bold', pad=20)
plt.xlabel('SHAP value (impact on model output)', fontsize=14)
plt.tight_layout()

beeswarm_hires_path = os.path.join(shap_output_dir, 'shap_beeswarm_plot_top50_hires.png')
plt.savefig(beeswarm_hires_path, dpi=600, bbox_inches='tight')
plt.close()
print(f"  ✅ 高分辨率蜂群图已保存: {beeswarm_hires_path}")

# ====================== 13. 可视化：特征重要性排名（突出显示RFE特征）======================
print("\n创建特征重要性排名图...")

# 图1：特征重要性排名（突出显示RFE特征）
fig, ax = plt.subplots(figsize=(14, 10))

# 绘制所有特征
colors = ['red' if feat in RFE_SELECTED_FEATURES else 'gray' 
          for feat in shap_importance_df['Feature'].head(50)]
sizes = [100 if feat in RFE_SELECTED_FEATURES else 30 
         for feat in shap_importance_df['Feature'].head(50)]

ax.barh(range(50), 
        shap_importance_df['Mean_Absolute_SHAP_Value'].head(50), 
        color=colors, alpha=0.6)

# 添加特征名称
ax.set_yticks(range(50))
ax.set_yticklabels(shap_importance_df['Feature'].head(50), fontsize=8)
ax.set_xlabel('Mean Absolute SHAP Value', fontsize=12)
ax.set_ylabel('Features', fontsize=12)
ax.set_title('Top 50 Feature Importance (Red = RFE Selected)', fontsize=14, fontweight='bold')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)

# 添加图例
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='red', alpha=0.6, label='RFE Selected (6 features)'),
    Patch(facecolor='gray', alpha=0.6, label='Other Features')
]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig(os.path.join(shap_output_dir, 'feature_importance_ranking.png'), dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✅ 特征重要性排名图已保存")

# 图2：RFE特征的排名条形图
fig, ax = plt.subplots(figsize=(10, 6))

rfe_sorted = rfe_ranking_df.sort_values('SHAP_Rank')
bars = ax.barh(rfe_sorted['RFE_Feature'], rfe_sorted['SHAP_Rank'], 
               color='steelblue', alpha=0.7, edgecolor='black')

# 添加排名标签
for i, (idx, row) in enumerate(rfe_sorted.iterrows()):
    ax.text(row['SHAP_Rank'] + 1, i, f"#{int(row['SHAP_Rank'])}", 
            va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Rank (out of 103 features)', fontsize=12)
ax.set_ylabel('RFE Selected Features', fontsize=12)
ax.set_title('SHAP Ranking of RFE Selected Features', fontsize=14, fontweight='bold')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)
ax.set_xlim(0, max(rfe_sorted['SHAP_Rank']) + 10)

plt.tight_layout()
plt.savefig(os.path.join(shap_output_dir, 'RFE_features_ranking.png'), dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✅ RFE特征排名图已保存")

# ====================== 14. 保存详细的SHAP值数据 ======================
print("\n保存详细SHAP数据...")

# 获取模型预测
with torch.no_grad():
    all_predictions = model(torch.tensor(X_scaled, dtype=torch.float32).to(device)).cpu().numpy().flatten()

# 创建完整的数据表
base_value = explainer.expected_value
if hasattr(base_value, "__len__") and len(base_value) == 1:
    base_value = base_value[0]

# 原始特征值
feature_values_df = pd.DataFrame(X_original, columns=features_103)

# SHAP值
shap_values_df = pd.DataFrame(shap_values, columns=[f'SHAP_{col}' for col in features_103])

# 预测信息
predictions_df = pd.DataFrame({
    'sample_index': range(len(X_scaled)),
    'prediction': all_predictions,
    'actual_value': y,
    'base_value': base_value
})

# 合并
combined_df = pd.concat([predictions_df, feature_values_df, shap_values_df], axis=1)
combined_path = os.path.join(shap_output_dir, 'all_features_shap_predictions.xlsx')
combined_df.to_excel(combined_path, index=False)
print(f"  ✅ 完整SHAP数据已保存: {combined_path}")

# ====================== 15. 生成汇总报告 ======================
print("\n生成汇总报告...")

report_path = os.path.join(shap_output_dir, 'SHAP_Analysis_Report.txt')
with open(report_path, 'w', encoding='utf-8') as f:
    f.write("="*80 + "\n")
    f.write("审稿人验证分析：103特征SHAP分析报告（增强版）\n")
    f.write("="*80 + "\n\n")
    
    f.write("分析目的:\n")
    f.write("  1. 验证RFE筛选的6个特征在全部103个特征中的重要性排名\n")
    f.write("  2. 生成Top 50特征的SHAP蜂群图\n")
    f.write("  3. 导出完整的可视化数据供进一步分析\n\n")
    
    f.write("数据信息:\n")
    f.write(f"  总特征数: 103\n")
    f.write(f"  样本数: {X_scaled.shape[0]}\n")
    f.write(f"  RFE筛选特征数: 6\n")
    f.write(f"  蜂群图展示特征数: 50\n\n")
    
    f.write("="*80 + "\n")
    f.write("关键结果：RFE筛选的6个特征在SHAP分析中的排名\n")
    f.write("="*80 + "\n\n")
    
    for result in rfe_ranking_results:
        f.write(f"特征: {result['RFE_Feature']}\n")
        f.write(f"  SHAP排名: {result['SHAP_Rank']} / 103\n")
        f.write(f"  平均绝对SHAP值: {result['Mean_Abs_SHAP']:.6f}\n")
        f.write(f"  相对重要性: {result['Relative_Importance_%']:.2f}%\n\n")
    
    f.write("="*80 + "\n")
    f.write("统计总结\n")
    f.write("="*80 + "\n\n")
    f.write(f"RFE的6个特征排名统计:\n")
    f.write(f"  最高排名: {min(ranks)}\n")
    f.write(f"  最低排名: {max(ranks)}\n")
    f.write(f"  平均排名: {np.mean(ranks):.1f}\n")
    f.write(f"  中位排名: {np.median(ranks):.1f}\n\n")
    
    f.write(f"排名分布:\n")
    f.write(f"  Top 10: {top_10_count} / 6 特征 ({top_10_count/6*100:.1f}%)\n")
    f.write(f"  Top 20: {top_20_count} / 6 特征 ({top_20_count/6*100:.1f}%)\n")
    f.write(f"  Top 30: {top_30_count} / 6 特征 ({top_30_count/6*100:.1f}%)\n\n")
    
    f.write("="*80 + "\n")
    f.write("蜂群图分析\n")
    f.write("="*80 + "\n\n")
    f.write(f"Top 50特征蜂群图已生成，展示了:\n")
    f.write(f"  - 每个特征对模型预测的影响方向（正/负）\n")
    f.write(f"  - 特征值的分布（颜色编码）\n")
    f.write(f"  - SHAP值的分布范围\n")
    f.write(f"  - 特征重要性排序\n\n")
    
    rfe_in_top50 = sum(1 for feat in RFE_SELECTED_FEATURES if feat in top50_features)
    f.write(f"RFE的6个特征中有 {rfe_in_top50} 个出现在Top 50中\n\n")
    
    f.write("="*80 + "\n")
    f.write("结论\n")
    f.write("="*80 + "\n\n")
    
    if np.mean(ranks) <= 20:
        f.write("✅ RFE筛选的6个特征在103个特征中表现优异！\n")
        f.write("   平均排名在Top 20以内，证明RFE筛选方法有效。\n")
    elif np.mean(ranks) <= 30:
        f.write("✓ RFE筛选的6个特征在103个特征中表现良好。\n")
        f.write("   平均排名在Top 30以内，RFE筛选方法合理。\n")
    else:
        f.write("⚠️ RFE筛选的特征排名较为分散。\n")
        f.write("   建议进一步分析特征筛选策略。\n")
    
    f.write("\n" + "="*80 + "\n")
    f.write("生成的文件清单\n")
    f.write("="*80 + "\n\n")
    f.write("📊 Excel数据文件:\n")
    f.write("  1. feature_importance_all_103.xlsx - 全部103个特征的SHAP重要性\n")
    f.write("  2. RFE_features_SHAP_ranking.xlsx - RFE的6个特征在103个中的排名\n")
    f.write("  3. all_features_shap_predictions.xlsx - 完整SHAP数据（所有样本）\n")
    f.write("  4. 🆕 beeswarm_plot_data_top50.xlsx - 蜂群图完整数据（可重绘）\n")
    f.write("  5. 🆕 beeswarm_summary_statistics_top50.xlsx - Top50特征统计摘要\n\n")
    
    f.write("📈 可视化图表:\n")
    f.write("  6. feature_importance_ranking.png - 特征重要性可视化（红色=RFE特征）\n")
    f.write("  7. RFE_features_ranking.png - RFE特征排名条形图\n")
    f.write("  8. 🆕 shap_beeswarm_plot_top50.png - Top50特征蜂群图\n")
    f.write("  9. 🆕 shap_beeswarm_plot_top50_hires.png - 高分辨率蜂群图（600 DPI）\n\n")
    
    f.write("📄 报告文件:\n")
    f.write("  10. SHAP_Analysis_Report.txt - 本报告\n")
    f.write("="*80 + "\n\n")
    
    f.write("使用说明:\n")
    f.write("  - Excel文件可用于进一步的数据分析和自定义可视化\n")
    f.write("  - 蜂群图PNG文件可直接用于论文和演示\n")
    f.write("  - 高分辨率版本适合期刊投稿要求\n")
    f.write("  - 所有原始数据已保存，支持完全重现分析结果\n")
    f.write("="*80 + "\n")

print(f"  ✅ 汇总报告已保存: {report_path}")

# ====================== 16. 最终输出 ======================
print("\n" + "="*80)
print("=== SHAP分析完成（增强版）===")
print("="*80)
print(f"所有结果已保存到：{shap_output_dir}")
print("\n📊 主要Excel数据文件:")
print(f"  1. RFE_features_SHAP_ranking.xlsx - RFE特征排名（⭐ 最重要）")
print(f"  2. feature_importance_all_103.xlsx - 全部特征重要性")
print(f"  3. beeswarm_plot_data_top50.xlsx - 蜂群图数据（⭐ 新增）")
print(f"  4. beeswarm_summary_statistics_top50.xlsx - 统计摘要（⭐ 新增）")
print(f"  5. all_features_shap_predictions.xlsx - 完整预测数据")
print("\n📈 可视化图表:")
print(f"  6. shap_beeswarm_plot_top50.png - 蜂群图（⭐ 新增）")
print(f"  7. shap_beeswarm_plot_top50_hires.png - 高清蜂群图（⭐ 新增）")
print(f"  8. feature_importance_ranking.png - 特征排名图")
print(f"  9. RFE_features_ranking.png - RFE排名图")
print("\n📄 报告文件:")
print(f"  10. SHAP_Analysis_Report.txt - 汇总报告")
print("\n" + "="*80)
print("🎯 审稿人验证完成！")
print("🆕 新增功能：Top 50特征蜂群图 + 完整可视化数据导出")
print("="*80)

🎯 审稿人验证程序：SHAP分析（103个特征）- 增强版
目的：计算103个特征的SHAP值，并查看RFE筛选的6个特征排名
新增：Top 50特征蜂群图 + 完整可视化数据导出

路径设置:
  模型路径: D:\博一下\manuscript3-机器学习\nc\nc修改9.27\10.15\10.21\1.4程序补充\验证分析_103特征
  SHAP输出路径: D:\博一下\manuscript3-机器学习\nc\nc修改9.27\10.15\10.21\1.4程序补充\SHAP分析_103特征1.17

🎯 RFE筛选的6个关键特征:
  1. ΔHmix
  2. mean_C1 temperature melting
  3. mean_S13 radii atomic (coordination number 12) (pm)
  4. mean_热导率W/(mk)
  5. var_C1 temperature melting
  6. var_热导率W/(mk)

加载数据...
  特征数量: 103
  ✅ 标准化器已加载

加载模型...
  ✅ 模型权重已加载

准备SHAP分析数据...
  样本数: 221
  特征数: 103

开始SHAP分析...
  这可能需要几分钟时间...
  背景样本数: 100
  ✅ SHAP解释器已创建
  正在计算所有 221 个样本的SHAP值...


  0%|          | 0/221 [00:00<?, ?it/s]

  ✅ SHAP值计算完成
  SHAP值形状: (221, 103)

计算特征重要性...
  ✅ 特征重要性已保存: D:\博一下\manuscript3-机器学习\nc\nc修改9.27\10.15\10.21\1.4程序补充\SHAP分析_103特征1.17\feature_importance_all_103.xlsx

🎯 关键结果：RFE筛选的6个特征在103个特征中的SHAP排名
✓ ΔHmix
  排名: 22 / 103
  平均绝对SHAP值: 0.095200
  相对重要性: 1.49%

✓ mean_C1 temperature melting
  排名: 100 / 103
  平均绝对SHAP值: 0.000000
  相对重要性: 0.00%

✓ mean_S13 radii atomic (coordination number 12) (pm)
  排名: 92 / 103
  平均绝对SHAP值: 0.000680
  相对重要性: 0.01%

✓ mean_热导率W/(mk)
  排名: 16 / 103
  平均绝对SHAP值: 0.110133
  相对重要性: 1.73%

✓ var_C1 temperature melting
  排名: 84 / 103
  平均绝对SHAP值: 0.005438
  相对重要性: 0.09%

✓ var_热导率W/(mk)
  排名: 70 / 103
  平均绝对SHAP值: 0.012054
  相对重要性: 0.19%

✅ RFE特征排名已保存: D:\博一下\manuscript3-机器学习\nc\nc修改9.27\10.15\10.21\1.4程序补充\SHAP分析_103特征1.17\RFE_features_SHAP_ranking.xlsx

📊 统计总结
RFE的6个特征排名统计:
  最高排名: 16
  最低排名: 100
  平均排名: 64.0
  中位排名: 77.0

排名分布:
  Top 10: 0 / 6 特征 (0.0%)
  Top 20: 1 / 6 特征 (16.7%)
  Top 30: 2 / 6 特征 (33.3%)

🆕 生成Top 50特征蜂群图及可视化数据
  Top 50特征已提取
  SHAP值矩阵形状: 